1. Weighted Sum(Linear Transformation)

A neuron first computes a weighted sum of its inputs.
because it has the forme 
    y = Wx+b 

This is exactly the same equation used in
    Linear Regression
    Matrix Multiplication
    Geometry
It cannot create curves. it only produces straight lines 

2.Non-linear Activation
Now we pass the weighted sum through an activation function.
for example RELU 

==>Every neuron in a neural network performs these two steps. weight sum and non-linear activation 

Why not use only the weighted sum?



In [11]:
import torch 
import torch.nn as nn 

#inputs 
x = torch.tensor([[2.,5.,1.]])

#linear layer (weighted sum)

linear = nn.Linear(3,2) # matrix (2 line ,3 colune)

#activation function 
relu = nn.ReLU()

#step1: weighted sum 

z= linear(x)
print("weighted sum :")
print(z)

#step 2 : non-linear activation 
y = relu(z)
print("\n after Relu")
print(y)

weighted sum :
tensor([[ 2.1914, -0.7160]], grad_fn=<AddmmBackward0>)

 after Relu
tensor([[2.1914, 0.0000]], grad_fn=<ReluBackward0>)


nn.MultiheadAttention is one of the most important layers in modern AI. It is the core building block of Transformers, which power models like:

ChatGPT
GPT-4
BERT
Vision Transformer

Unlike a convolutional layer (which only looks at nearby pixels) or an RNN (which processes one token at a time), Multi-Head Attention allows every token to look at every other token simultaneously.

Why was it invented?

Suppose you have the sentence

The cat sat on the mat because it was tired.

Question:

What does "it" refer to?

Possible words:

The
cat
sat
on
the
mat
because
it
was
tired

Humans immediately know

it
↓

cat

A neural network also needs to discover this relationship.

That is exactly what attention does.

Step 1. Input Embeddings

Every word is converted into a vector.

"The"

↓

[0.25, -0.41, 0.82, ...]
"cat"

↓

[-0.53, 1.20, -0.17, ...]

Suppose

Sentence

The cat sat

Embeddings

The

↓

[0.2 0.1 0.8]

Cat

↓

[0.9 0.5 0.3]

Sat

↓

[0.4 0.6 0.7]

These vectors are the input.

Step 2. Create Query, Key and Value

Every input vector is transformed into three new vectors.

For every word:

Embedding

↓

Query (Q)

↓

Key (K)

↓

Value (V)

These are computed using three learnable matrices:

Q = XWq

K = XWk

V = XWv

where

X = input embeddings
Wq = learned query weights
Wk = learned key weights
Wv = learned value weights
What are Query, Key, and Value?

Think of a library.

Books have:

    Book

    ↓

    Title

That is the Key.

When you search

Machine Learning

that search is the Query.

The actual content of the book is the Value.

In attention

                Query

                ↓

                What am I looking for?

                Key

                ↓

                What information do I contain?

                Value

                ↓

                What information should I send?
Step 3. Compare Query with all Keys

Suppose the word is

it

Its query is compared with every key.

                    it

                    ↓

                    The

                    ↓

                    Cat

                    ↓

                    Sat

                    ↓

                    Mat

                    ↓

                    Tired

This comparison is a dot product. Q*transpose(K)

arge value

High similarity

Small value

Low similarity

Example

   Word	    Score
    The	    0.2
    Cat	    9.5
    Sat	    2.0
    Mat	    0.8
    Tired	6.1

Clearly

    Cat

    ↓

    Highest score
Step 4. Softmax

These scores become probabilities.

Example

                Before

                0.2

                9.5

                2

                6

                After Softmax

                0.01

                0.72

                0.03

                0.24

                Now

                72%

                ↓

                Cat

The model mostly focuses on cat.

Step 5. Weighted Sum

Each Value vector is multiplied by its attention weight.

                Value(The)

                ×

                0.01

                +

                Value(Cat)

                ×

                0.72

                +

                Value(Sat)

                ×

                0.03

                ...

                ↓

                New representation

The output vector for "it" now contains information gathered from the other relevant words.


self attention formula 
 attention(Q,K,V) = softmax((Q*transpose(K))/racine(dk))*V


 What is Multi-Head Attention?

Instead of computing one attention, we compute many attentions in parallel.

Example
        Input

        ↓

        Head 1

        ↓

        Head 2

        ↓

        Head 3

        ↓

        Head 4

        ↓

        Concatenate

        ↓

        Output

Example

Sentence

The red car is fast.

            Head 1

            Focuses on grammar

            car ← is

            Head 2

            Focuses on color

            car ← red

            Head 3

            Focuses on meaning

            car ← fast

            Head 4

            Focuses on position

Every head learns independently.


Why Multiple Heads?

Suppose we have only one head.

        Sentence

        ↓

        One relationship

We lose other important information.

With eight heads

            Sentence

            ↓

            Grammar

            ↓

            Meaning

            ↓

            Location

            ↓

            Objects

            ↓

            Actions

            ↓

            Long-distance relationships

            ↓

            Numbers

            ↓

            Names


    Input Embeddings
        │
        ▼
 ┌─────────────────┐
 │ Linear Layers   │
 │ Create Q K V    │
 └─────────────────┘
        │
        ▼
 ┌────────────┐
 │ Head 1     │
 ├────────────┤
 │ Head 2     │
 ├────────────┤
 │ Head 3     │
 ├────────────┤
 │ Head 4     │
 └────────────┘
        │
Concatenate
        │
        ▼
Linear Layer
        │
        ▼
Output

In [5]:
import torch
import torch.nn as nn

# Multi-head attention layer
mha = nn.MultiheadAttention(
    embed_dim=8,      # Size of each embedding vector
    num_heads=2,      # Number of attention heads
    batch_first=True  # Input shape: (batch, seq_len, embed_dim)
)

# Batch size = 1
# Sequence length = 4
# Embedding dimension = 8
x = torch.randn(1, 4, 8)
print(x)
# Self-attention
output, attention_weights = mha(x, x, x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)

tensor([[[-0.9852,  0.0281, -0.3003,  1.0543,  1.1434, -0.6443,  0.5859,
          -0.1200],
         [ 0.7342,  1.2584,  0.4613,  0.0130,  1.2136, -0.6910, -0.1350,
          -2.2807],
         [-0.5850,  1.5639,  0.2283, -1.0518,  0.1438,  0.6385, -0.0701,
           1.2089],
         [-0.4681, -0.7014, -0.7926,  1.4919, -0.6345,  1.3316, -1.1120,
          -0.8477]]])
Input shape: torch.Size([1, 4, 8])
Output shape: torch.Size([1, 4, 8])
Attention weights shape: torch.Size([1, 4, 4])


Why is the attention weight shape (1, 4, 4)?
1 = batch size
The first 4 = each of the 4 tokens acts as a Query
The second 4 (keys)= each Query attends to all 4 Keys

So the attention matrix might look like:
| Query \ Key | Token 1 | Token 2 | Token 3 | Token 4 |
| ----------- | ------: | ------: | ------: | ------: |
| Token 1     |    0.60 |    0.20 |    0.10 |    0.10 |
| Token 2     |    0.05 |    0.80 |    0.10 |    0.05 |
| Token 3     |    0.15 |    0.25 |    0.50 |    0.10 |
| Token 4     |    0.20 |    0.10 |    0.15 |    0.55 |


Sentiment Analysis (Positive or Negative Review)
build an AI that classifies movie reviews.
exemple 
    "I absolutely loved this movie because the acting was amazing."
goal : positive 

step 1 : tolenization split into the words tokens 
step 2 : word embeddings each word becomes a vector 
step 3 : create Q,k,V Every embedding produces three vectors.
step 4 : compute attention
    Now consider the word "loved"
        The model asks
        "Which words help me understand loved?"
            | Word       | Attention Score |
            | ---------- | --------------: |
            | I          |            0.03 |
            | absolutely |            0.24 |
            | loved      |            0.31 |
            | movie      |            0.07 |
            | because    |            0.02 |
            | acting     |            0.11 |
            | amazing    |            0.22 |


            loved
            ↓
            I              3%
            absolutely    24%
            loved         31%
            movie          7%
            acting        11%
            amazing       22%
    The model learns that "absolutely" and "amazing" are very important for understanding "loved".

Step 5: Build a New Representation
    The output vector for "loved" becomes
            0.03 × Value(I)
            +
            0.24 × Value(absolutely)
            +
            0.31 × Value(loved)
            +
            ...
            ↓
            New Vector

            Now "loved" contains information from the whole sentence.


the input value is -4
| Activation          | Output                                                                    |
| ------------------- | ------------------------------------------------------------------------- |
| ReLU                | 0                                                                         |
| LeakyReLU (0.1)     | -0.4                                                                      |
| PReLU (learned 0.3) | -1.2                                                                      |
| ReLU6               | 0                                                                         |
| RReLU               | Random value between about -0.5 and -1.3 (depending on the sampled slope) |



In [6]:
import torch 
import torch.nn as nn 

x = torch.tensor([-5., -2., 0., 2., 8.])

activations = {
    "ReLU":nn.ReLU(),
    "PReLU": nn.PReLU(),
    "ReLU6": nn.ReLU6(),
    "RReLU": nn.RReLU(lower=0.1, upper=0.3)
    
}

for name, act in activations.items():
    print(f"{name}:")
    print(act(x))
    print()

ReLU:
tensor([0., 0., 0., 2., 8.])

PReLU:
tensor([-1.2500, -0.5000,  0.0000,  2.0000,  8.0000],
       grad_fn=<PreluKernelBackward0>)

ReLU6:
tensor([0., 0., 0., 2., 6.])

RReLU:
tensor([-0.5897, -0.4395,  0.0000,  2.0000,  8.0000])



nn.SELU (Scaled Exponential Linear Unit)

SELU was designed to make neural networks self-normalizing.

The idea is that during training, the activations automatically stay close to:
    Mean ≈ 0
    Variance ≈ 1

    This helps stabilize training in very deep networks.
Positive values become slightly larger because of the scaling factor λ.

In [7]:
import torch
import torch.nn as nn

x = torch.tensor([-3., -1., 0., 2.])
selu = nn.SELU()
print(selu(x))

tensor([-1.6706, -1.1113,  0.0000,  2.1014])


2. nn.CELU (Continuously Differentiable ELU)
CELU is an improved version of ELU.
ELU has a small discontinuity in its derivative around zero.
CELU removes this problem.

nn.GELU (Gaussian Error Linear Unit)

This is one of the most important activation functions today.

It is used in:

GPT
BERT
Vision Transformer
Large Language Models

instead os 
    Negative
    ↓
    Discard

GELU asks
"How likely is this value to be useful?"
Small values are partially kept, not abruptly removed.

In [8]:
x = torch.tensor([-3., -1., 0., 1., 3.])

gelu = nn.GELU()

print(gelu(x))

tensor([-0.0041, -0.1587,  0.0000,  0.8413,  2.9959])


nn.Sigmoid

nn.SiLU (Sigmoid Linear Unit)

Also called Swish.

Discovered by researchers at Google.

Used in:

EfficientNet
YOLOv8
Modern CNNs


Why use SiLU?
Advantages
Smooth.
Better gradients.
Often higher accuracy than ReLU.

In [9]:
x = torch.tensor([-3.,-1.,0.,2.])

silu = nn.SiLU()

print(silu(x))

tensor([-0.1423, -0.2689,  0.0000,  1.7616])


nn.Mish

In [10]:
x = torch.tensor([-3.,-1.,0.,2.])

mish = nn.Mish()

print(mish(x))

tensor([-0.1456, -0.3034,  0.0000,  1.9440])


Visual Comparison
suppose input is -2 

| Activation | Output (approx.) |
| ---------- | ---------------- |
| ReLU       | 0                |
| ELU        | -0.86            |
| SELU       | -1.52            |
| CELU       | -0.86            |
| GELU       | -0.05            |
| Sigmoid    | 0.12             |
| SiLU       | -0.24            |
| Mish       | -0.25            |



In [11]:
import torch
import torch.nn as nn

x = torch.tensor([-3., -1., 0., 1., 3.])

activations = {
    "SELU": nn.SELU(),
    "CELU": nn.CELU(),
    "GELU": nn.GELU(),
    "Sigmoid": nn.Sigmoid(),
    "SiLU": nn.SiLU(),
    "Mish": nn.Mish()
}

for name, act in activations.items():
    print(f"{name}:")
    print(act(x))
    print()

SELU:
tensor([-1.6706, -1.1113,  0.0000,  1.0507,  3.1521])

CELU:
tensor([-0.9502, -0.6321,  0.0000,  1.0000,  3.0000])

GELU:
tensor([-0.0041, -0.1587,  0.0000,  0.8413,  2.9959])

Sigmoid:
tensor([0.0474, 0.2689, 0.5000, 0.7311, 0.9526])

SiLU:
tensor([-0.1423, -0.2689,  0.0000,  0.7311,  2.8577])

Mish:
tensor([-0.1456, -0.3034,  0.0000,  0.8651,  2.9865])



| Activation       | Best Use                                     | Advantages                                      | Drawbacks                                                        |
| ---------------- | -------------------------------------------- | ----------------------------------------------- | ---------------------------------------------------------------- |
| **SELU**         | Deep fully connected networks                | Self-normalizing activations                    | Requires specific initialization (LeCun Normal) and AlphaDropout |
| **CELU**         | When you want ELU with smoother derivatives  | Smooth optimization                             | Less commonly used                                               |
| **GELU**         | Transformers (GPT, BERT, ViT)                | Excellent performance and smooth gradients      | More computationally expensive than ReLU                         |
| **Sigmoid**      | Binary classification output layer           | Produces values between 0 and 1 (probabilities) | Suffers from vanishing gradients in hidden layers                |
| **SiLU (Swish)** | Modern CNNs (EfficientNet, YOLO)             | Smooth, strong empirical performance            | Slightly slower than ReLU                                        |
| **Mish**         | Experimental and some computer vision models | Very smooth, can improve accuracy               | Higher computational cost and less common in production          |


nn.Softplus 
Softplus is a smooth version of ReLU.
Instead of abruptly changing from 0 to x, Softplus transitions smoothly.
    Why use Softplus?
    Advantages
    Smooth.
    Differentiable everywhere.
    Never has dead neurons.
    Disadvantages
    Slower than ReLU.
    Less commonly used in CNNs.

In [12]:
import torch
import torch.nn as nn

x = torch.tensor([-3., -1., 0., 1., 3.])

softplus = nn.Softplus()

print(softplus(x))

tensor([0.0486, 0.3133, 0.6931, 1.3133, 3.0486])


nn.Softshrink
Softshrink removes values close to zero.
Unlike Hardshrink,
it reduces values gradually.

nn.Softsign
Softsign behaves similarly to Tanh but is computationally simpler.

In [13]:
x = torch.tensor([-4., -1., 0., 1., 4.])

soft = nn.Softsign()

print(soft(x))

tensor([-0.8000, -0.5000,  0.0000,  0.5000,  0.8000])


TANH 
Advantages
Centered around zero.
Better than Sigmoid for hidden layers.
nn.Tanhshrink

nn.Threshold
Threshold simply compares each value against a chosen threshold.

In [14]:
import torch
import torch.nn as nn

x = torch.tensor([-2., 0., 1., 5.])

thr = nn.Threshold(
    threshold=1,
    value=-100
)

print(thr(x))

tensor([-100., -100., -100.,    5.])


nn.GLU (Gated Linear Unit)
GLU is completely different from the previous activations.
It uses a gate to decide how much information should pass through.
Think of it like a water valve.

In [15]:
import torch
import torch.nn as nn

# Last dimension must be even because GLU splits it in half
x = torch.randn(2, 6)

glu = nn.GLU(dim=1)
y = glu(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)

Input shape : torch.Size([2, 6])
Output shape: torch.Size([2, 3])


The output has half as many features because the input is split into two equal parts.
GLU is used in:

Some Transformer variants
Gated CNNs
Language models
Speech recognition

The gating mechanism helps the network learn which information to keep and which to suppress, similar in spirit to the gates in LSTMs.

| Activation     | Output Range               | Main Idea                                 | Typical Use                              |    |                      |
| -------------- | -------------------------- | ----------------------------------------- | ---------------------------------------- | -- | -------------------- |
| **Softplus**   | (0, ∞)                     | Smooth approximation of ReLU              | Bayesian deep learning, VAEs             |    |                      |
| **Softshrink** | Depends on input           | Gradually shrinks values toward zero      | Denoising, sparse models                 |    |                      |
| **Softsign**   | (-1, 1)                    | Smooth saturation using (x/(1+            | x                                        | )) | Lightweight networks |
| **Tanh**       | (-1, 1)                    | Zero-centered activation                  | RNNs, LSTMs, GRUs                        |    |                      |
| **Tanhshrink** | Unbounded                  | Computes (x - \tanh(x))                   | Mostly research                          |    |                      |
| **Threshold**  | Depends on threshold/value | Replaces values below a threshold         | Rule-based filtering                     |    |                      |
| **GLU**        | Depends on input           | Learns a gate to control information flow | Transformers, language and speech models |    |                      |


| Activation        | Popularity                          | Where you'll see it                       |
| ----------------- | ----------------------------------- | ----------------------------------------- |
| ⭐⭐⭐⭐⭐ **Tanh**    | Very common                         | RNNs, LSTMs, GRUs                         |
| ⭐⭐⭐⭐ **Softplus** | Moderately common                   | Probabilistic models, VAEs                |
| ⭐⭐⭐⭐ **GLU**      | Common in specialized architectures | Transformers, gated CNNs, speech models   |
| ⭐⭐ **Softsign**   | Occasional                          | Lightweight or experimental networks      |
| ⭐ **Softshrink**  | Rare                                | Signal processing, sparse representations |
| ⭐ **Threshold**   | Rare                                | Custom architectures, preprocessing       |
| ⭐ **Tanhshrink**  | Very rare                           | Mostly research experiments               |
